In [5]:
import pandas as pd
import numpy as np
import requests

In [2]:
INPUT_CSV = "guwahati_shillong_edges.csv"
OUTPUT_CSV = "guwahati_shillong_edges.csv"

In [3]:
ELEVATION_URL = "https://api.open-meteo.com/v1/elevation"
BATCH_SIZE = 100

In [4]:
def fetch_elevations_batch(lats, lons):
    """Fetches elevation for a batch of coordinates in one API call."""
    lat_str = ",".join(str(x) for x in lats)
    lon_str = ",".join(str(x) for x in lons)
    resp = requests.get(ELEVATION_URL, params={"latitude": lat_str, "longitude": lon_str}, timeout=30)
    resp.raise_for_status()
    return resp.json()["elevation"]
 
 
def fill_elevation():
    df = pd.read_csv(INPUT_CSV)
    n = len(df)
    print(f"Filling elevation for {n} corridor edges...")
 
    # We need elevation at BOTH the start and end point of every edge.
    # Build one combined list of points so we can batch the API calls
    # efficiently instead of calling once per point.
    all_lats = list(df["start_lat"]) + list(df["end_lat"])
    all_lons = list(df["start_lon"]) + list(df["end_lon"])
 
    all_elevations = []
    for i in range(0, len(all_lats), BATCH_SIZE):
        batch_lats = all_lats[i:i + BATCH_SIZE]
        batch_lons = all_lons[i:i + BATCH_SIZE]
        batch_elevations = fetch_elevations_batch(batch_lats, batch_lons)
        all_elevations.extend(batch_elevations)
        print(f"  Fetched {min(i + BATCH_SIZE, len(all_lats))}/{len(all_lats)} points")
 
    start_elevations = all_elevations[:n]
    end_elevations = all_elevations[n:]
 
    df["elevation_start_m"] = start_elevations
    df["elevation_end_m"] = end_elevations
    df["elevation_min"] = df[["elevation_start_m", "elevation_end_m"]].min(axis=1)
    df["elevation_max"] = df[["elevation_start_m", "elevation_end_m"]].max(axis=1)
    df["elevation_avg"] = df[["elevation_start_m", "elevation_end_m"]].mean(axis=1)
 
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved to {OUTPUT_CSV}")
    return df
 
 
if __name__ == "__main__":
    fill_elevation()

Filling elevation for 175 corridor edges...
  Fetched 100/350 points
  Fetched 200/350 points
  Fetched 300/350 points
  Fetched 350/350 points

Saved to guwahati_shillong_edges.csv


In [6]:
def calculate_slope():
    df = pd.read_csv(INPUT_CSV)
 
    height_diff = df["elevation_max"] - df["elevation_min"]
 
    # length is already in meters in the OSMnx export
    # avoid divide-by-zero for any zero-length edges
    safe_length = df["length"].replace(0, np.nan)
 
    slope_radians = np.arctan(height_diff / safe_length)
    df["slope_deg"] = np.degrees(slope_radians).fillna(0).round(2)
 
    df.to_csv(OUTPUT_CSV, index=False)
 
    print(f"Slope calculated for {len(df)} edges")
    print(f"Slope range: {df['slope_deg'].min()}° to {df['slope_deg'].max()}°")
    print(f"Average slope: {df['slope_deg'].mean():.2f}°")
    print(f"\nSaved to {OUTPUT_CSV}")
    return df
 
 
if __name__ == "__main__":
    calculate_slope()

Slope calculated for 175 edges
Slope range: 0.0° to 45.91°
Average slope: 2.74°

Saved to guwahati_shillong_edges.csv
